# RAG-lite: Gutenberg BM25 Pipeline

This notebook shows how to build a simple RAG-style pipeline without any LLM. It consists of three steps:

1. **Ingest & chunk** local Gutenberg corpus books
2. **Build** a BM25 retrieval index over the resulting chunks
3. **Query** the BM25 index and assemble a RAG-style prompt

## Section 1 — Ingest & Chunk Books

Discovers text from `.txt` files, cleans and chunks each one in parallel using a local process pool, and collects the combined corpus into an in-memory `corpus` list.

In [ ]:
import os
import concurrent.futures as cf
from collections import Counter
from pathlib import Path

data_dir = Path("data")       # directory containing Gutenberg .txt files
workers = os.cpu_count()      # number of local worker processes


In [ ]:
def book_id_from_path(path: Path) -> str:
    return path.stem

In [ ]:
def clean_and_chunk_book(local_filename: str, book_id: str):
    """
    Read a local Gutenberg .txt file, clean it, and chunk it into ~1000-character
    segments for RAG-style use. Imports are kept inside the function so it can be
    shipped to a worker process cleanly (matches the original TaskVine worker shape).

    Returns:
        list[dict]: one dict per chunk (book_id, chunk_id, total_chunks,
                    relative_position, text, chunk_length, n_chars, n_words, preview)
    """
    import re
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    # --- 1. Read full file ---
    with open(local_filename, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    # --- 2. Strip Gutenberg boilerplate (best-effort) ---
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG",
        "*** START OF THE PROJECT GUTENBERG",
        "***START OF THE PROJECT GUTENBERG",
        "*END*THE SMALL PRINT",  # older texts
    ]
    for marker in start_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[idx + len(marker):]
            break

    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG",
        "*** END OF THE PROJECT GUTENBERG",
        "***END OF THE PROJECT GUTENBERG",
    ]
    for marker in end_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
            break

    # --- 3. Basic normalization ---
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    text = re.sub(r" +", " ", text)
    text = text.strip()

    # --- 4. Chunking ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_text(text)
    total_chunks = len(chunks)

    # --- 5. Build result records ---
    results = []
    for i, chunk in enumerate(chunks):
        words = re.findall(r"\b\w+\b", chunk)
        n_words = len(words)

        relative_position = i / (total_chunks - 1) if total_chunks > 1 else 0.0
        preview = chunk[:160].replace("\n", " ")

        results.append({
            "book_id": book_id,
            "chunk_id": i,
            "total_chunks": total_chunks,
            "relative_position": relative_position,
            "text": chunk,
            "chunk_length": len(chunk),
            "n_chars": len(chunk),
            "n_words": n_words,
            "preview": preview,
        })

    return results


In [ ]:
book_paths = sorted(data_dir.glob("*.txt"))
if not book_paths:
    raise RuntimeError(f"No .txt files found in {data_dir}")

print("Found books:")
for p in book_paths:
    print(f" - {p.name} ({p.stat().st_size} bytes)")

book_ids = [book_id_from_path(p) for p in book_paths]
print("\nBook IDs:", book_ids)
print(f"\nLocal workers: {workers}")

corpus = []
with cf.ProcessPoolExecutor(max_workers=workers) as executor:
    futures = {}
    for path in book_paths:
        book_id = book_id_from_path(path)
        fut = executor.submit(clean_and_chunk_book, str(path), book_id)
        futures[fut] = {"book_id": book_id, "path": str(path), "file_size": path.stat().st_size}

    total_tasks = len(futures)
    print(f"\nSubmitted {total_tasks} local chunking tasks")

    completed = 0
    for fut in cf.as_completed(futures):
        completed += 1
        meta = futures[fut]
        book_id = meta["book_id"]

        try:
            book_chunks = fut.result()
            corpus.extend(book_chunks)
            print(f"[{completed}/{total_tasks}] \u2713 book {book_id} -> {len(book_chunks)} chunks")
        except Exception as e:
            print(f"[{completed}/{total_tasks}] \u2717 book {book_id} FAILED: {e}")

print("\nAll tasks done.")
print(f"Total chunks collected: {len(corpus)}")

# ---- corpus stays in memory for Section 2 (no JSON file written) ----

book_counts = Counter(c["book_id"] for c in corpus)
print("\nChunks per book:")
for b, cnt in sorted(book_counts.items()):
    print(f"  {b}: {cnt} chunks")

chunk_lengths = [c["chunk_length"] for c in corpus]
if chunk_lengths:
    print("\nChunk length stats:")
    print(f"  min: {min(chunk_lengths)}")
    print(f"  max: {max(chunk_lengths)}")
    print(f"  avg: {sum(chunk_lengths) / len(chunk_lengths):.1f}")


## Section 2 — Build the BM25 Retrieval Index

Converts the in-memory corpus previously created into LangChain Documents and builds a BM25Retriever.

In [ ]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever  # uses rank-bm25 under the hood

k = 4  # default top-k chunks the retriever returns per query


In [ ]:
documents = [
    Document(
        page_content=c["text"],
        metadata={
            "book_id": c["book_id"],
            "chunk_id": c["chunk_id"],
            "relative_position": c["relative_position"],
            "n_words": c["n_words"],
        },
    )
    for c in corpus
]
print(f"Built {len(documents)} Documents from corpus")

retriever = BM25Retriever.from_documents(documents)
retriever.k = k
print(f"\u2713 BM25 retriever ready (k={k})")


## Section 3 — Query the BM25 Rerieval Index

Runs one or more queries against the BM25 retrieval index and prints the retrieved chunks plus the assembled context and question prompt.

In [ ]:
def rag_query(retriever, query: str, k: int = 4):
    """
    RAG-style helper using the modern LangChain 'invoke' interface.
    Returns (results, prompt); also prints a readable summary.
    """
    retriever.k = k
    results = retriever.invoke(query)

    print("\n" + "=" * 70)
    print(f"RAG query: {query!r}")
    print("=" * 70)
    print(f"Retrieved {len(results)} chunks:\n")

    for i, doc in enumerate(results, 1):
        meta = doc.metadata
        pos_pct = f"{100 * meta.get('relative_position', 0.0):.1f}%"
        print(f"[{i}] book_id={meta['book_id']}  chunk_id={meta['chunk_id']}  pos={pos_pct}")
        preview = doc.page_content[:200].replace("\n", " ")
        print(f"    {preview}...")
        print()

    context = "\n\n".join(
        f"[book {doc.metadata['book_id']} | chunk {doc.metadata['chunk_id']}] {doc.page_content}"
        for doc in results
    )

    prompt = f"""You are a helpful assistant answering questions about classic literature.

Use ONLY the following context to answer the question. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}
Answer:"""

    return results, prompt


In [ ]:
queries = [
    "What happens when Alice falls down the rabbit hole?",
    "What does Hamlet mean when he says 'To be or not to be'?",
]
k = 4  # top-k chunks per query at retrieval time

for query in queries:
    _, prompt = rag_query(retriever, query, k)
    print(prompt)
    print()
